## Logistic Regression with PySpark

In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.appName('log_reg').getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/20 17:47:12 WARN Utils: Your hostname, aditya-HP-Laptop-15s-eq1xxx, resolves to a loopback address: 127.0.1.1; using 10.103.210.123 instead (on interface wlo1)
26/06/20 17:47:12 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/20 17:47:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
!curl https://raw.githubusercontent.com/apache/spark/refs/heads/master/data/mllib/sample_libsvm_data.txt >> sample_libsvm_data.txt

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  102k  100  102k    0     0   157k      0 --:--:-- --:--:-- --:--:--  158k


In [5]:
df = spark.read.format('libsvm').load('sample_libsvm_data.txt')

26/06/20 17:49:04 WARN LibSVMFileFormat: 'numFeatures' option not specified, determining the number of features by going though the input. If you know the number in advance, please specify it via 'numFeatures' option to avoid the extra scan.
                                                                                

In [6]:
df.printSchema()

root
 |-- label: double (nullable = true)
 |-- features: vector (nullable = true)



In [7]:
from pyspark.ml.classification import LogisticRegression

In [8]:
lr = LogisticRegression()

model = lr.fit(df)

summary = model.summary

In [9]:
summary.predictions.show()

+-----+--------------------+--------------------+--------------------+----------+
|label|            features|       rawPrediction|         probability|prediction|
+-----+--------------------+--------------------+--------------------+----------+
|  0.0|(692,[127,128,129...|[20.3777627514875...|[0.99999999858729...|       0.0|
|  1.0|(692,[158,159,160...|[-21.114014198852...|[6.76550380011201...|       1.0|
|  1.0|(692,[124,125,126...|[-23.743613234684...|[4.87842678711891...|       1.0|
|  1.0|(692,[152,153,154...|[-19.192574012724...|[4.62137287296030...|       1.0|
|  1.0|(692,[151,152,153...|[-20.125398874706...|[1.81823629111716...|       1.0|
|  0.0|(692,[129,130,131...|[20.4890549504206...|[0.99999999873608...|       0.0|
|  1.0|(692,[158,159,160...|[-21.082940212796...|[6.97903542836027...|       1.0|
|  1.0|(692,[99,100,101,...|[-19.622713503561...|[3.00582577442810...|       1.0|
|  0.0|(692,[154,155,156...|[21.1594863606525...|[0.99999999935352...|       0.0|
|  0.0|(692,[127

In [10]:
from pyspark.mllib.evaluation import MulticlassMetrics

In [11]:
model.evaluate(df)

In [12]:
pred_and_labels = model.evaluate(df)

In [13]:
pred_and_labels.predictions.show()

+-----+--------------------+--------------------+--------------------+----------+
|label|            features|       rawPrediction|         probability|prediction|
+-----+--------------------+--------------------+--------------------+----------+
|  0.0|(692,[127,128,129...|[20.3777627514875...|[0.99999999858729...|       0.0|
|  1.0|(692,[158,159,160...|[-21.114014198852...|[6.76550380011201...|       1.0|
|  1.0|(692,[124,125,126...|[-23.743613234684...|[4.87842678711891...|       1.0|
|  1.0|(692,[152,153,154...|[-19.192574012724...|[4.62137287296030...|       1.0|
|  1.0|(692,[151,152,153...|[-20.125398874706...|[1.81823629111716...|       1.0|
|  0.0|(692,[129,130,131...|[20.4890549504206...|[0.99999999873608...|       0.0|
|  1.0|(692,[158,159,160...|[-21.082940212796...|[6.97903542836027...|       1.0|
|  1.0|(692,[99,100,101,...|[-19.622713503561...|[3.00582577442810...|       1.0|
|  0.0|(692,[154,155,156...|[21.1594863606525...|[0.99999999935352...|       0.0|
|  0.0|(692,[127

In [14]:
pred_and_labels = pred_and_labels.predictions.select("label", "prediction")

In [15]:
pred_and_labels.show()

+-----+----------+
|label|prediction|
+-----+----------+
|  0.0|       0.0|
|  1.0|       1.0|
|  1.0|       1.0|
|  1.0|       1.0|
|  1.0|       1.0|
|  0.0|       0.0|
|  1.0|       1.0|
|  1.0|       1.0|
|  0.0|       0.0|
|  0.0|       0.0|
|  1.0|       1.0|
|  0.0|       0.0|
|  0.0|       0.0|
|  1.0|       1.0|
|  0.0|       0.0|
|  1.0|       1.0|
|  0.0|       0.0|
|  0.0|       0.0|
|  1.0|       1.0|
|  1.0|       1.0|
+-----+----------+
only showing top 20 rows


#### Evaluation

In [16]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

In [17]:
evalu = BinaryClassificationEvaluator(rawPredictionCol="prediction", labelCol="label")

In [19]:
eval_multi = MulticlassClassificationEvaluator(predictionCol="prediction", 
                                               labelCol="label",
                                              metricName="accuracy")

In [20]:
acc = evalu.evaluate(pred_and_labels)

In [21]:
acc

1.0

### Logistic Regression: Titanic Dataset

In [22]:
!curl https://raw.githubusercontent.com/markumreed/colab_pyspark/main/titanic.csv >> titanic.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 60302  100 60302    0     0  56326      0  0:00:01  0:00:01 --:--:-- 56357


In [23]:
from pyspark.sql import SparkSession

In [24]:
spark = SparkSession.builder.appName('titanic').getOrCreate()

26/06/20 18:05:03 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [25]:
df = spark.read.csv('titanic.csv', inferSchema=True, header=True)

In [26]:
df.printSchema()

root
 |-- PassengerId: integer (nullable = true)
 |-- Survived: integer (nullable = true)
 |-- Pclass: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Age: double (nullable = true)
 |-- SibSp: integer (nullable = true)
 |-- Parch: integer (nullable = true)
 |-- Ticket: string (nullable = true)
 |-- Fare: double (nullable = true)
 |-- Cabin: string (nullable = true)
 |-- Embarked: string (nullable = true)



In [27]:
df.columns

['PassengerId',
 'Survived',
 'Pclass',
 'Name',
 'Sex',
 'Age',
 'SibSp',
 'Parch',
 'Ticket',
 'Fare',
 'Cabin',
 'Embarked']

In [29]:
data = df.select([
 'Survived',
 'Pclass',
 'Sex',
 'Age',
 'SibSp',
 'Parch',
 'Fare',
 'Embarked'
])

In [30]:
data.head()

Row(Survived=0, Pclass=3, Sex='male', Age=22.0, SibSp=1, Parch=0, Fare=7.25, Embarked='S')

In [31]:
data_final = data.na.drop()

#### Categorical data with pyspark

In [32]:
from pyspark.ml.feature import (VectorAssembler, VectorIndexer, 
                        OneHotEncoder, StringIndexer)

In [33]:
gender_indexer = StringIndexer(inputCol='Sex', outputCol='SexIndex')
gender_encoder = OneHotEncoder(inputCol='SexIndex', outputCol='SexVec')

embark_indexer = StringIndexer(inputCol='Embarked', outputCol='EmbarkIndex')
embark_encoder = OneHotEncoder(inputCol='EmbarkIndex', outputCol='EmbarkVec')

In [34]:
assembler = VectorAssembler(inputCols=['Pclass', 'SexVec', 'Age', 'SibSp',
                                      'Parch', 'Fare', 'EmbarkVec'], outputCol='features')

In [35]:
from pyspark.ml.classification import LogisticRegression

#### Pipelines 

In [36]:
from pyspark.ml import Pipeline

In [37]:
log_reg = LogisticRegression(featuresCol='features', labelCol='Survived')

In [38]:
pipeline = Pipeline(stages=[
    gender_indexer,
    embark_indexer,
    gender_encoder,
    embark_encoder,
    assembler,
    log_reg
])

In [39]:
train, test = data_final.randomSplit([0.7, 0.3], seed=42)

In [40]:
model_fit = pipeline.fit(train)
res = model_fit.transform(test)

26/06/20 18:32:47 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [41]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

In [42]:
evalu = BinaryClassificationEvaluator(rawPredictionCol='prediction',
                                     labelCol='Survived')

In [43]:
res.select('Survived', 'prediction').show()

+--------+----------+
|Survived|prediction|
+--------+----------+
|       0|       1.0|
|       0|       1.0|
|       0|       1.0|
|       0|       1.0|
|       0|       1.0|
|       0|       0.0|
|       0|       1.0|
|       0|       1.0|
|       0|       1.0|
|       0|       1.0|
|       0|       1.0|
|       0|       0.0|
|       0|       0.0|
|       0|       0.0|
|       0|       0.0|
|       0|       1.0|
|       0|       0.0|
|       0|       0.0|
|       0|       0.0|
|       0|       1.0|
+--------+----------+
only showing top 20 rows


In [44]:
auc = evalu.evaluate(res)

In [45]:
auc

0.7747561675272518